In [26]:
# ============================================================
# MOTOR FINANCIERO DE RECOMENDACION BANCARIA
# ============================================================
#
# PIPELINE COMPLETO
#
# 1. Candidate Generation
# 2. Ranking Unificado
# 3. Expansion Cliente-Producto
# 4. Modelo Utilidad
# 5. Ajuste Riesgo
# 6. Elegibilidad
# 7. Financial Reranking
# 8. Top K Final
#
# ============================================================

import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# ============================================================
# CONFIG
# ============================================================

TOP_CANDIDATOS = 10
TOP_K_FINAL = 3

# pesos score final

W_PROP = 0.40
W_UTIL = 0.45
W_RISK = 0.15

# ============================================================
# 1. DATAFRAME CLIENTES
# ============================================================

df_clientes = pd.DataFrame({

    'id_cliente': [1,2,3,4,5],

    'segmento': [
        'pyme',
        'persona',
        'empresa',
        'pyme',
        'gobierno'
    ],

    'ingresos': [
        15000000,
        4000000,
        60000000,
        12000000,
        100000000
    ],

    'pd': [
        0.05,
        0.75,
        0.02,
        0.15,
        0.01
    ],

    'mora_dias': [
        0,
        120,
        0,
        30,
        0
    ],

    'endeudamiento': [
        0.30,
        0.90,
        0.25,
        0.55,
        0.10
    ],

    'utilidad_empresa': [
        5000000,
        -500000,
        30000000,
        2000000,
        40000000
    ],

    'tiene_cta_corriente': [
        1,
        0,
        1,
        1,
        1
    ]
})
df_clientes

,id_cliente,segmento,ingresos,pd,mora_dias,endeudamiento,utilidad_empresa,tiene_cta_corriente
0,1,pyme,15000000,0.05,0,0.30,5000000,1
1,2,persona,4000000,0.75,120,0.90,-500000,0
2,3,empresa,60000000,0.02,0,0.25,30000000,1
3,4,pyme,12000000,0.15,30,0.55,2000000,1
4,5,gobierno,100000000,0.01,0,0.10,40000000,1


In [27]:
# ============================================================
# 2. PRODUCTOS HISTORICOS APERTURADOS
# ============================================================

# utilidad REAL observada futura

df_aperturas = pd.DataFrame({

    'id_cliente': [

        1,1,
        2,
        3,
        4,
        5,5

    ],

    'producto': [

        'tarjeta_credito',
        'cdt',

        'credito_personal',

        'cash_management',

        'leasing',

        'cdt',
        'cash_management'
    ],

    'utilidad_12m': [

        800000,
        100000,

        -4000000,

        2500000,

        -1500000,

        500000,
        4000000
    ]
})
df_aperturas


,id_cliente,producto,utilidad_12m
0,1,tarjeta_credito,800000
1,1,cdt,100000
2,2,credito_personal,-4000000
3,3,cash_management,2500000
4,4,leasing,-1500000
5,5,cdt,500000
6,5,cash_management,4000000


In [28]:
# ============================================================
# 3. RECOMENDADOR HISTORICO
# ============================================================

# Esto representa:
#
# productos recomendados por el sistema
#
# NO productos aperturados
#
# ============================================================

df_recomendaciones = pd.DataFrame({

    'id_cliente': [

        1,1,1,1,1,
        2,2,2,2,2,
        3,3,3,3,3,
        4,4,4,4,4,
        5,5,5,5,5

    ],

    'producto': [

        'tarjeta_credito',
        'cdt',
        'leasing',
        'cash_management',
        'credito_personal',

        'credito_personal',
        'tarjeta_credito',
        'cdt',
        'leasing',
        'cash_management',

        'cash_management',
        'leasing',
        'cdt',
        'tarjeta_credito',
        'credito_personal',

        'leasing',
        'credito_personal',
        'cash_management',
        'cdt',
        'tarjeta_credito',

        'cash_management',
        'cdt',
        'leasing',
        'credito_personal',
        'tarjeta_credito'
    ],

    'score_propension': [

        0.92,
        0.80,
        0.60,
        0.50,
        0.40,

        0.95,
        0.60,
        0.30,
        0.20,
        0.10,

        0.90,
        0.75,
        0.40,
        0.30,
        0.20,

        0.85,
        0.70,
        0.60,
        0.40,
        0.20,

        0.95,
        0.85,
        0.50,
        0.30,
        0.20
    ],

    'rank_propension': [

        1,2,3,4,5,
        1,2,3,4,5,
        1,2,3,4,5,
        1,2,3,4,5,
        1,2,3,4,5
    ]
})
df_recomendaciones


,id_cliente,producto,score_propension,rank_propension
0,1,tarjeta_credito,0.92,1
1,1,cdt,0.80,2
2,1,leasing,0.60,3
3,1,cash_management,0.50,4
4,1,credito_personal,0.40,5
5,2,credito_personal,0.95,1
6,2,tarjeta_credito,0.60,2
7,2,cdt,0.30,3
8,2,leasing,0.20,4
9,2,cash_management,0.10,5


In [29]:
# ============================================================
# 4. EXPANSION CLIENTE-PRODUCTO
# ============================================================

df_modelo = (
    df_recomendaciones
    .merge(
        df_clientes,
        on='id_cliente',
        how='left'
    )
)
df_modelo


,id_cliente,producto,score_propension,rank_propension,segmento,ingresos,pd,mora_dias,endeudamiento,utilidad_empresa,tiene_cta_corriente
0,1,tarjeta_credito,0.92,1,pyme,15000000,0.05,0,0.30,5000000,1
1,1,cdt,0.80,2,pyme,15000000,0.05,0,0.30,5000000,1
2,1,leasing,0.60,3,pyme,15000000,0.05,0,0.30,5000000,1
3,1,cash_management,0.50,4,pyme,15000000,0.05,0,0.30,5000000,1
4,1,credito_personal,0.40,5,pyme,15000000,0.05,0,0.30,5000000,1
5,2,credito_personal,0.95,1,persona,4000000,0.75,120,0.90,-500000,0
6,2,tarjeta_credito,0.60,2,persona,4000000,0.75,120,0.90,-500000,0
7,2,cdt,0.30,3,persona,4000000,0.75,120,0.90,-500000,0
8,2,leasing,0.20,4,persona,4000000,0.75,120,0.90,-500000,0
9,2,cash_management,0.10,5,persona,4000000,0.75,120,0.90,-500000,0


In [30]:
# ============================================================
# 5. AGREGAR UTILIDAD OBSERVADA
# ============================================================

df_modelo = (
    df_modelo
    .merge(
        df_aperturas,
        on=['id_cliente', 'producto'],
        how='left'
    )
)
df_modelo


,id_cliente,producto,score_propension,rank_propension,segmento,ingresos,pd,mora_dias,endeudamiento,utilidad_empresa,tiene_cta_corriente,utilidad_12m
0,1,tarjeta_credito,0.92,1,pyme,15000000,0.05,0,0.30,5000000,1,800000.0
1,1,cdt,0.80,2,pyme,15000000,0.05,0,0.30,5000000,1,100000.0
2,1,leasing,0.60,3,pyme,15000000,0.05,0,0.30,5000000,1,NaN
3,1,cash_management,0.50,4,pyme,15000000,0.05,0,0.30,5000000,1,NaN
4,1,credito_personal,0.40,5,pyme,15000000,0.05,0,0.30,5000000,1,NaN
5,2,credito_personal,0.95,1,persona,4000000,0.75,120,0.90,-500000,0,-4000000.0
6,2,tarjeta_credito,0.60,2,persona,4000000,0.75,120,0.90,-500000,0,NaN
7,2,cdt,0.30,3,persona,4000000,0.75,120,0.90,-500000,0,NaN
8,2,leasing,0.20,4,persona,4000000,0.75,120,0.90,-500000,0,NaN
9,2,cash_management,0.10,5,persona,4000000,0.75,120,0.90,-500000,0,NaN


In [32]:
df_aperturas

,id_cliente,producto,utilidad_12m
0,1,tarjeta_credito,800000
1,1,cdt,100000
2,2,credito_personal,-4000000
3,3,cash_management,2500000
4,4,leasing,-1500000
5,5,cdt,500000
6,5,cash_management,4000000


In [31]:
# si no aperturo:
# utilidad = 0

df_modelo['utilidad_12m'] = (
    df_modelo['utilidad_12m']
    .fillna(0)
)

# ============================================================
# 6. FEATURE ENGINEERING
# ============================================================

df_modelo['pd_x_ingresos'] = (
    df_modelo['pd'] *
    df_modelo['ingresos']
)

df_modelo['deuda_x_pd'] = (
    df_modelo['endeudamiento'] *
    df_modelo['pd']
)

df_modelo['mora_x_pd'] = (
    df_modelo['mora_dias'] *
    df_modelo['pd']
)

df_modelo['score_x_pd'] = (
    df_modelo['score_propension'] *
    (1 - df_modelo['pd'])
)

# flag credito

productos_credito = [

    'credito_personal',
    'leasing',
    'tarjeta_credito'
]

df_modelo['es_credito'] = (
    df_modelo['producto']
    .isin(productos_credito)
).astype(int)
df_modelo


,id_cliente,producto,score_propension,rank_propension,segmento,ingresos,pd,mora_dias,endeudamiento,utilidad_empresa,tiene_cta_corriente,utilidad_12m,pd_x_ingresos,deuda_x_pd,mora_x_pd,score_x_pd,es_credito
0,1,tarjeta_credito,0.92,1,pyme,15000000,0.05,0,0.30,5000000,1,800000.0,750000.0,0.0150,0.0,0.8740,1
1,1,cdt,0.80,2,pyme,15000000,0.05,0,0.30,5000000,1,100000.0,750000.0,0.0150,0.0,0.7600,0
2,1,leasing,0.60,3,pyme,15000000,0.05,0,0.30,5000000,1,0.0,750000.0,0.0150,0.0,0.5700,1
3,1,cash_management,0.50,4,pyme,15000000,0.05,0,0.30,5000000,1,0.0,750000.0,0.0150,0.0,0.4750,0
4,1,credito_personal,0.40,5,pyme,15000000,0.05,0,0.30,5000000,1,0.0,750000.0,0.0150,0.0,0.3800,1
5,2,credito_personal,0.95,1,persona,4000000,0.75,120,0.90,-500000,0,-4000000.0,3000000.0,0.6750,90.0,0.2375,1
6,2,tarjeta_credito,0.60,2,persona,4000000,0.75,120,0.90,-500000,0,0.0,3000000.0,0.6750,90.0,0.1500,1
7,2,cdt,0.30,3,persona,4000000,0.75,120,0.90,-500000,0,0.0,3000000.0,0.6750,90.0,0.0750,0
8,2,leasing,0.20,4,persona,4000000,0.75,120,0.90,-500000,0,0.0,3000000.0,0.6750,90.0,0.0500,1
9,2,cash_management,0.10,5,persona,4000000,0.75,120,0.90,-500000,0,0.0,3000000.0,0.6750,90.0,0.0250,0


In [34]:
# ============================================================
# 7. VARIABLES MODELO
# ============================================================

TARGET = 'utilidad_12m'

FEATURES = [

    'segmento',
    'producto',
    'score_propension',
    'rank_propension',

    'ingresos',
    'pd',
    'mora_dias',
    'endeudamiento',
    'utilidad_empresa',
    'tiene_cta_corriente',

    'pd_x_ingresos',
    'deuda_x_pd',
    'mora_x_pd',
    'score_x_pd',

    'es_credito'
]

CAT_FEATURES = [

    'segmento',
    'producto'
]

X = df_modelo[FEATURES]
y = df_modelo[TARGET]

X


,segmento,producto,score_propension,rank_propension,ingresos,pd,mora_dias,endeudamiento,utilidad_empresa,tiene_cta_corriente,pd_x_ingresos,deuda_x_pd,mora_x_pd,score_x_pd,es_credito
0,pyme,tarjeta_credito,0.92,1,15000000,0.05,0,0.30,5000000,1,750000.0,0.0150,0.0,0.8740,1
1,pyme,cdt,0.80,2,15000000,0.05,0,0.30,5000000,1,750000.0,0.0150,0.0,0.7600,0
2,pyme,leasing,0.60,3,15000000,0.05,0,0.30,5000000,1,750000.0,0.0150,0.0,0.5700,1
3,pyme,cash_management,0.50,4,15000000,0.05,0,0.30,5000000,1,750000.0,0.0150,0.0,0.4750,0
4,pyme,credito_personal,0.40,5,15000000,0.05,0,0.30,5000000,1,750000.0,0.0150,0.0,0.3800,1
5,persona,credito_personal,0.95,1,4000000,0.75,120,0.90,-500000,0,3000000.0,0.6750,90.0,0.2375,1
6,persona,tarjeta_credito,0.60,2,4000000,0.75,120,0.90,-500000,0,3000000.0,0.6750,90.0,0.1500,1
7,persona,cdt,0.30,3,4000000,0.75,120,0.90,-500000,0,3000000.0,0.6750,90.0,0.0750,0
8,persona,leasing,0.20,4,4000000,0.75,120,0.90,-500000,0,3000000.0,0.6750,90.0,0.0500,1
9,persona,cash_management,0.10,5,4000000,0.75,120,0.90,-500000,0,3000000.0,0.6750,90.0,0.0250,0


In [35]:
y

0      800000.0
1      100000.0
2           0.0
3           0.0
4           0.0
5    -4000000.0
6           0.0
7           0.0
8           0.0
9           0.0
10    2500000.0
11          0.0
12          0.0
13          0.0
14          0.0
15   -1500000.0
16          0.0
17          0.0
18          0.0
19          0.0
20    4000000.0
21     500000.0
22          0.0
23          0.0
24          0.0
Name: utilidad_12m, dtype: float64

In [17]:
qwe

NameError: name 'qwe' is not defined

In [ ]:
# ============================================================
# 8. TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.30,
    random_state=42
)

# ============================================================
# 9. ENTRENAMIENTO MODELO UTILIDAD
# ============================================================

model_utilidad = CatBoostRegressor(

    iterations=500,
    learning_rate=0.03,
    depth=6,

    loss_function='RMSE',

    verbose=100
)

model_utilidad.fit(

    X_train,
    y_train,

    cat_features=CAT_FEATURES
)

# ============================================================
# 10. EVALUACION
# ============================================================

preds = model_utilidad.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print("MAE:", mae)

from sklearn.metrics import r2_score
r2 = r2_score(y_test, preds)
print("R2:", r2)

0:	learn: 1529218.5483719	total: 367ms	remaining: 3m 3s
100:	learn: 720988.3640283	total: 10.2s	remaining: 40.3s
200:	learn: 338886.8871529	total: 18.1s	remaining: 27s
300:	learn: 162301.6100161	total: 27.1s	remaining: 17.9s
400:	learn: 78796.7158894	total: 34.3s	remaining: 8.48s
499:	learn: 42398.5271583	total: 41s	remaining: 0us

MAE
91842.60185430685


In [38]:
# ============================================================
# 11. IMPORTANCIA VARIABLES
# ============================================================

importance = pd.DataFrame({

    'feature': FEATURES,

    'importance': model_utilidad.get_feature_importance()

})

importance = importance.sort_values(

    'importance',
    ascending=False
)

print('\nIMPORTANCIA VARIABLES')
print(importance)



IMPORTANCIA VARIABLES
                feature  importance
3       rank_propension   19.544567
2      score_propension   17.962132
13           score_x_pd   16.777221
1              producto    7.925422
12            mora_x_pd    6.683538
7         endeudamiento    5.354877
6             mora_dias    4.738714
5                    pd    4.364255
4              ingresos    4.103588
0              segmento    3.477887
11           deuda_x_pd    3.389963
10        pd_x_ingresos    2.018010
8      utilidad_empresa    1.690241
14           es_credito    1.498976
9   tiene_cta_corriente    0.470608


In [ ]:
# ============================================================
# 12. FUNCION ELEGIBILIDAD
# ============================================================

def aplicar_elegibilidad(df):

    df = df.copy()

    df['elegible'] = 1

    # --------------------------------------------------------
    # CASH REQUIERE CUENTA CORRIENTE
    # --------------------------------------------------------

    mask = (

        (df['producto'] == 'cash_management') &
        (df['tiene_cta_corriente'] == 0)
    )

    df.loc[mask, 'elegible'] = 0

    # --------------------------------------------------------
    # PD ALTO
    # --------------------------------------------------------

    mask = (

        (df['pd'] > 0.70) &
        (df['es_credito'] == 1)
    )

    df.loc[mask, 'elegible'] = 0

    # --------------------------------------------------------
    # MORA ALTA
    # --------------------------------------------------------

    mask = (

        (df['mora_dias'] > 90) &
        (df['es_credito'] == 1)
    )

    df.loc[mask, 'elegible'] = 0

    # --------------------------------------------------------
    # UTILIDAD NEGATIVA EMPRESA
    # --------------------------------------------------------

    mask = (

        (df['utilidad_empresa'] < 0) &
        (df['es_credito'] == 1)
    )

    df.loc[mask, 'elegible'] = 0

    return df


In [ ]:

# ============================================================
# 13. FUNCION MOTOR FINANCIERO
# ============================================================

def motor_financiero_cliente(

    row_cliente,
    df_recomendaciones_cliente,
    model_utilidad,
    top_k=3
):

    # --------------------------------------------------------
    # EXPANSION CLIENTE-PRODUCTO
    # --------------------------------------------------------

    df = (
        df_recomendaciones_cliente
        .copy()
    )

    # agregar variables cliente

    for col in row_cliente.index:

        df[col] = row_cliente[col]

    # --------------------------------------------------------
    # FEATURE ENGINEERING
    # --------------------------------------------------------

    df['pd_x_ingresos'] = (
        df['pd'] *
        df['ingresos']
    )

    df['deuda_x_pd'] = (
        df['endeudamiento'] *
        df['pd']
    )

    df['mora_x_pd'] = (
        df['mora_dias'] *
        df['pd']
    )

    df['score_x_pd'] = (
        df['score_propension'] *
        (1 - df['pd'])
    )

    df['es_credito'] = (
        df['producto']
        .isin(productos_credito)
    ).astype(int)

    # --------------------------------------------------------
    # UTILIDAD ESPERADA
    # --------------------------------------------------------

    df['utilidad_pred'] = model_utilidad.predict(
        df[FEATURES]
    )

    # --------------------------------------------------------
    # AJUSTE RIESGO
    # --------------------------------------------------------

    df['utilidad_ajustada'] = (

        df['utilidad_pred'] *

        (1 - df['pd'])
    )

    # --------------------------------------------------------
    # SCORE FINAL
    # --------------------------------------------------------

    df['score_final'] = (

        W_PROP *
        df['score_propension']

        +

        W_UTIL *
        (
            df['utilidad_ajustada'] /
            (
                np.abs(
                    df['utilidad_ajustada']
                ).max() + 1
            )
        )

        -

        W_RISK *
        df['pd']
    )

    # --------------------------------------------------------
    # ELEGIBILIDAD
    # --------------------------------------------------------

    df = aplicar_elegibilidad(df)

    # --------------------------------------------------------
    # ELIMINAR NO RENTABLES
    # --------------------------------------------------------

    df = df.query(

        "utilidad_ajustada > 0"
    )

    # --------------------------------------------------------
    # ELIMINAR NO ELEGIBLES
    # --------------------------------------------------------

    df = df.query(

        "elegible == 1"
    )

    # --------------------------------------------------------
    # RERANKING FINANCIERO
    # --------------------------------------------------------

    df = df.sort_values(

        'score_final',
        ascending=False
    )

    # --------------------------------------------------------
    # TOP K FINAL
    # --------------------------------------------------------

    df = df.head(top_k)

    return df


In [ ]:
# ============================================================
# 14. EJEMPLO PRODUCCION
# ============================================================

# cliente nuevo

row_cliente = pd.Series({

    'id_cliente': 999,

    'segmento': 'pyme',

    'ingresos': 25000000,

    'pd': 0.10,

    'mora_dias': 0,

    'endeudamiento': 0.40,

    'utilidad_empresa': 5000000,

    'tiene_cta_corriente': 1
})
row_cliente


In [ ]:
# recomendaciones iniciales
# candidate generation

df_recomendaciones_cliente = pd.DataFrame({

    'producto': [

        'credito_personal',
        'leasing',
        'tarjeta_credito',
        'cash_management',
        'cdt'
    ],

    'score_propension': [

        0.95,
        0.90,
        0.88,
        0.70,
        0.60
    ],

    'rank_propension': [

        1,
        2,
        3,
        4,
        5
    ]
})
df_recomendaciones_cliente


In [ ]:
# ============================================================
# 15. MOTOR FINAL
# ============================================================

top_final = motor_financiero_cliente(

    row_cliente=row_cliente,

    df_recomendaciones_cliente=df_recomendaciones_cliente,

    model_utilidad=model_utilidad,

    top_k=3
)

print('\nTOP FINAL RECOMENDACIONES')
display(top_final)
